# K-Fold Cross Validation — VGG-19

Validação cruzada com **K=10** para o modelo VGG-19.

Adam (lr=0.000125), FC: 1024-256, Dropout: 0.48/0.25

In [ ]:
import os, torch, numpy as np, torch.nn as nn, torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, SubsetRandomSampler
from sklearn.model_selection import KFold
from ignite.engine import Engine, Events
from ignite.handlers import EarlyStopping
from ignite.metrics import Accuracy, Loss
import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv()

DATASET_PATH = os.getenv("DATASET_PATH", "/caminho/para/DADOS-DIVIDIDOS")
PRETRAINED_WEIGHTS = os.getenv("VGG19_PRETRAINED", "/caminho/para/models/VggNet19-model-96.pth")
FEATURE_EXTRACT = True
K_FOLDS = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")


## 1. Dataset

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224), transforms.CenterCrop(224),
        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}
full_dataset_train = datasets.ImageFolder(os.path.join(DATASET_PATH, 'train'), data_transforms['train'])
full_dataset_val = datasets.ImageFolder(os.path.join(DATASET_PATH, 'train'), data_transforms['val'])
dataset_size = len(full_dataset_train)
print(f"Dataset: {dataset_size} imagens | Classes: {full_dataset_train.classes}")


## 2. K-Fold Cross Validation

In [ ]:
def set_parameter_requires_grad(model, fe):
    if fe:
        for p in model.parameters(): p.requires_grad = False

def create_model():
    model = models.vgg19(pretrained=False)
    set_parameter_requires_grad(model, FEATURE_EXTRACT)
    state_dict = torch.load(PRETRAINED_WEIGHTS, map_location=device)
    del state_dict["classifier.6.weight"]; del state_dict["classifier.6.bias"]
    model.load_state_dict(state_dict, strict=False)
    num_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(nn.Dropout(0.4751), nn.Linear(num_features, 1024), nn.ReLU(), nn.Dropout(0.2508), nn.Linear(1024, 256), nn.ReLU(), nn.Linear(256, 2))
    return model.to(device)

kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold_results = []

for fold, (train_ids, val_ids) in enumerate(kfold.split(range(dataset_size))):
    print(f"FOLD {fold + 1}/{K_FOLDS}")
    train_sampler = SubsetRandomSampler(train_ids)
    val_sampler = SubsetRandomSampler(val_ids)
    train_loader = DataLoader(full_dataset_train, batch_size=128, sampler=train_sampler, num_workers=4)
    val_loader = DataLoader(full_dataset_val, batch_size=128, sampler=val_sampler, num_workers=4)

    model = create_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.000125)

    def train_step(engine, batch):
        model.train(); x, y = batch[0].to(device), batch[1].to(device)
        optimizer.zero_grad(); out = model(x); loss = criterion(out, y)
        loss.backward(); optimizer.step(); return loss.item(), out, y
    def eval_step(engine, batch):
        model.eval()
        with torch.no_grad(): x, y = batch[0].to(device), batch[1].to(device); return model(x), y

    trainer = Engine(train_step); evaluator = Engine(eval_step)
    Accuracy().attach(evaluator, "accuracy")
    def score_fn(e): return e.state.metrics["accuracy"]
    evaluator.add_event_handler(Events.COMPLETED, EarlyStopping(patience=10, score_function=score_fn, trainer=trainer))

    best_acc = [0.0]
    @trainer.on(Events.EPOCH_COMPLETED)
    def log(engine):
        evaluator.run(val_loader)
        acc = evaluator.state.metrics["accuracy"]
        best_acc[0] = max(best_acc[0], acc)

    trainer.run(train_loader, max_epochs=50)
    fold_results.append(best_acc[0])
    print(f"  Best Accuracy: {best_acc[0]:.4f}")


## 3. Resultados

In [ ]:
print("RESULTADOS K-FOLD")
for i, acc in enumerate(fold_results):
    print(f"  Fold {i+1}: {acc:.4f}")
print(f"Media: {np.mean(fold_results):.4f} +/- {np.std(fold_results):.4f}")

plt.figure(figsize=(10, 5))
plt.bar(range(1, K_FOLDS+1), fold_results, color='#3498db', alpha=0.8)
plt.axhline(y=np.mean(fold_results), color='#e74c3c', linestyle='--')
plt.title('VGG-19 - Acuracia por Fold', fontsize=14, fontweight='bold')
plt.xlabel('Fold'); plt.ylabel('Acuracia')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
